# LODissea project
## A journey through European(a) cultural heritage database

---

### Abstract 
Europeana is really the place to discover Europe's digital cultural heritage? This exploratory research investigates whether Europeana functions effectively as a collaborative shared space for digital cultural heritage across Europe. Specifically, we examine the scale and distribution of active data providers, analyzing who they are, how they are distributed geographically and in percentage terms, what institutional categories they represent, and the overall structural quality of their shared data. The selected country for this exploration are: Italy, Germany, Spain, Portugal, France, Netherdlands.

---

### Preliminary

#### Shared europeana key
The Europeana API key is loaded from a `.env` file (`EUROPEANA_API_KEY=...`). Get a free key at https://api.europeana.eu/.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# API KEY: reads EUROPEANA_API_KEY from a local .env file
load_dotenv()
EUROPEANA_API_KEY = os.environ.get("EUROPEANA_API_KEY", "")
if not EUROPEANA_API_KEY:
    print("WARNING: EUROPEANA_API_KEY not set.\n"
          "Create a .env file with: EUROPEANA_API_KEY=your_key_here\n"
          "The notebook will fall back to cached CSV/JSON files where available.")

EUROPEANA_SEARCH_URL = "https://api.europeana.eu/record/v2/search.json"
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "csv").mkdir(exist_ok=True)
(DATA_DIR / "json").mkdir(exist_ok=True)
REQUEST_DELAY = 0.3


#### Shared color palette
All country-level visualizations use this single color mapping for consistency.

In [3]:
# Country colors (reused across all visualizations)
COUNTRY_COLORS = {
    "netherlands": "#a180ad",
    "france":      "#1f7f95",
    "portugal":    "#f4a64e",
    "italy":       "#90BE6D",
    "germany":     "#feda15",
    "spain":       "#bb521f",
}

CATEGORY_COLORS = {
    "audiovisual/film archive": "#80CBC4",
    "art/history museum": "#9FA8DA",
    "natural history/science institution": "#CE93D8",
    "library/archive": "#90CAF9",
    "academic/research institution": "#FFCC80",
    "media/broadcast organization": "#EF9A9A",
    "government/administrative body": "#E8A0BE",
    "other": "#CFCFCF",
    "unresolved": "#9E9E9E",
}

# ISO-code variant for RQ1 visualizations (uses iso_code column)
ISO_TO_LOWER = {
    "IT": "italy", "DE": "germany", "NL": "netherlands",
    "PT": "portugal", "ES": "spain", "FR": "france",
}
COUNTRY_COLORS_ISO = {iso: COUNTRY_COLORS[lower] for iso, lower in ISO_TO_LOWER.items()}

print("Country color palette:")
for name, color in COUNTRY_COLORS.items():
    print(f"  {name:<15} {color}")


Country color palette:
  netherlands     #a180ad
  france          #1f7f95
  portugal        #f4a64e
  italy           #90BE6D
  germany         #feda15
  spain           #bb521f


---

## RQ01 Does the number of digital heritage and providers in Europeana reflect GLAM density or cultural expenditure (% GDP)?
### RsQ01 The number of Europeana providers reflect the number of GLAM institutions?
We began our inquiry by asking whether the volume of cultural heritage shared on Europeana reflects the true distribution of physical GLAM institutions across each country. Through this exploratory sub-question, we sought to understand whether Europeana functions as an equitable digital representation of national cultural heritage, and whether the number of active data providers scales proportionally with the underlying physical infrastructure. We interrogated Europeana data to find the number of providers for each country and Wikidata to find the number of GLAM institution.

### RsQ02 The volume of cultural heritage items shared via Europeana is proportional to the investment in culture done by each country?
Our second sub-question investigate further the relationship between investement and Europeana precence. With this exploration we are trying to understand if the selected country invest in sharing their digitaized cultural heritage on the European(a) platform. The data to answer this question was obtain from Eurostat and Europeana. 

### RsQ03 Does the number of active provider and volume of cultural heritage shared on Europeana correlate positevely to cultural investment? 
The third sub-question try to provide a summative exploration by observing the correlation between investments, Europeana providers to GLAM institutions rate and volume of Europeana data.

#### Install dependencies and import

In [4]:
# Install libraries

!pip install SPARQLWrapper rdflib eurostat plotly seaborn

In [5]:
# Import libraries
import requests
import json
import time
import pandas as pd
import eurostat
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from SPARQLWrapper import SPARQLWrapper, JSON

#### Mapping each country 
We mapped each country to a standardized dictionary of ISO codes, English names, and Wikidata QIDs was essential to establish a common relational key across four distinct data ecosystems (Eurostat, Europeana, Wikidata, and national censuses).

In [6]:
# Mapping countries
countries = {
    'IT': {'name_en': 'Italy','wd_id': 'Q38'},
    'DE': {'name_en': 'Germany', 'wd_id': 'Q183'},
    'NL': {'name_en': 'Netherlands', 'wd_id': 'Q55'},
    'PT': {'name_en': 'Portugal', 'wd_id': 'Q45'},
    'ES': {'name_en': 'Spain', 'wd_id': 'Q29'},
    'FR': {'name_en': 'France','wd_id': 'Q142'}
}

df_countries = pd.DataFrame.from_dict(countries, orient='index').reset_index()
df_countries.rename(columns={'index': 'iso_code'}, inplace=True)
df_countries

,iso_code,name_en,wd_id
0,IT,Italy,Q38
1,DE,Germany,Q183
2,NL,Netherlands,Q55
3,PT,Portugal,Q45
4,ES,Spain,Q29
5,FR,France,Q142


#### Querying the Europeana API for n° of cultural heritage items and n° of providers for each country
Data is collected via the Europeana Search REST API (`/record/v2/search.json`). Two sequential queries are executed per ISO 3166-1 alpha-2 country code:
* **Item Volume Extraction**: Sets parameter `rows=0` to retrieve the aggregate `totalResults` count per country.
* **Provider Facet Extraction**: Uses `profile='facets'`, `facet='DATA_PROVIDER'`, and `f.DATA_PROVIDER.facet.limit=1500` to extract unique contributing institutions without downloading record-level metadata.

*Note: A 0.5-second rate-limiting delay (`time.sleep`) is implemented to adhere to API request thresholds. Error handling defaults missing or failed requests to `0`.*


*Note: The europeana total items per country count accounts for Tier 0 entries as well, despite them not being visible on the website.*

In [7]:
# Europeana Data & Providers
API_KEY = EUROPEANA_API_KEY

europeana_data = []

for iso, info in countries.items():
    url = "https://api.europeana.eu/record/v2/search.json"
    params = {
        "wskey": API_KEY,
        "query": "*",
        "qf": f"COUNTRY:{info["name_en"].lower()}",
        "rows": 0,
        "profile": "facets",
        "facet": "DATA_PROVIDER",
        "f.DATA_PROVIDER.facet.limit": 1500
    }

    try:
        response = requests.get(url, params=params, timeout=20)

        if response.status_code == 200:
            resp_json = response.json()
            totale = resp_json.get("totalResults", 0)
            facets = resp_json.get("facets", [])

            provider_count = 0
            for facet in facets:
                if facet.get("name") == "DATA_PROVIDER":
                    provider_count = len(facet.get("fields", []))
                    break

            europeana_data.append({"iso_code": iso, "europeana_total": totale, "europeana_providers": provider_count})

        else:
            errore_msg = response.json().get("error", response.text)
            print(f"   Error HTTP {response.status_code}: {errore_msg}")
            europeana_data.append({"iso_code": iso, "europeana_total": 0, "europeana_providers": 0})

    except Exception as e:
        print(f"   Connection error for {info["name_en"]}: {e}")
        europeana_data.append({"iso_code": iso, "europeana_total": 0, "europeana_providers": 0})

    time.sleep(0.5)

df_europeana = pd.DataFrame(europeana_data).sort_values(by="europeana_total", ascending=False, ignore_index=True)
df_europeana


,iso_code,europeana_total,europeana_providers
0,NL,9204845,104
1,DE,8701240,375
2,ES,6581724,275
3,FR,4724898,50
4,IT,1832376,158
5,PT,139858,40


#### Querying the Eurostat database for expenditure for culture
To obtain macroeconomic spending metrics, the remote Eurostat database is queried directly into a Pandas DataFrame using the eurostat Python library.

* **Dataset Ingested**: General Government expenditure by function (`gov_10a_exp`).

* **Filtering Criteria**: Data is filtered to isolate General Government sector expenditure (`sector = 'S13'`) within the Culture function (`cofog99 = 'GF08'`), expressed as a percentage of national GDP (`'unit = 'PC_GDP'`).

* **Output**: Returns a filtered DataFrame containing the 2022 expenditure percentages for the selected Member States, which are mapped to ISO alpha-2 country codes under the column culture_expenditure_gdp_2022


In [8]:
# Public expenditure for culture (% of GDP)
from pathlib import Path
import os

csv_path = Path("data/csv/eurostat_culture_expenditure.csv")
if csv_path.exists():
    print("Loading Eurostat data from cache...")
    df_exp_filtered = pd.read_csv(csv_path)
else:
    print("Fetching Eurostat data...")
    df_exp = eurostat.get_data_df("gov_10a_exp")

    df_exp_filtered = df_exp[
        (df_exp["sector"] == "S13") & #General government
        (df_exp["unit"] == "PC_GDP") & #Percentage of GDP
        (df_exp["cofog99"] == "GF08") & #Culture sector
        (df_exp["na_item"] == "TE") &
        (df_exp["geo\\TIME_PERIOD"].isin(countries.keys()))
    ][["geo\\TIME_PERIOD", "2022"]].rename(columns={"geo\\TIME_PERIOD": "iso_code", "2022": "culture_expenditure_gpd_2022"})
    
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_exp_filtered.to_csv(csv_path, index=False)

df_exp_filtered


Loading Eurostat data from cache...


,iso_code,culture_expenditure_gpd_2022
0,DE,1.0
1,ES,1.2
2,FR,1.4
3,IT,0.9
4,NL,1.1
5,PT,0.9


#### Querying the remote Wikidata SPARQL endpoint for count of GLAM institutions
To establish a baseline count of physical GLAM institutions, the Wikidata Query Service endpoint (`https://query.wikidata.org/sparql`) is queried using `SPARQLWrapper`. 

* **Entity Classes Queried**: Museums (`wd:Q33506`), Libraries (`wd:Q7075`), Archives (`wd:Q166118`), and Art Galleries (`wd:Q1007870`).
* **Batch Processing**: Country QIDs are batched in groups of 3 (`BATCH_SIZE = 3`) using the `wdt:P17` (country) property to optimize query execution and prevent endpoint timeouts.
* **Output**: Returns distinct entity counts grouped by country QID, which are mapped back to ISO alpha-2 codes.

In [9]:
# Wikidata GLAM institutes for countries
from pathlib import Path
import os

csv_path = Path("data/csv/wikidata_glam_institutes.csv")
if csv_path.exists():
    print("Loading Wikidata GLAM data from cache...")
    df_wikidata = pd.read_csv(csv_path)
else:
    print("Fetching Wikidata GLAM data...")
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.agent = "info-viz-student-project/1.0 (mailto:nome.cognome@studio.unibo.it)"
    sparql.setReturnFormat(JSON)
    
    countries_list = [(iso, info['wd_id'], info['name_en']) for iso, info in countries.items()]
    
    BATCH_SIZE = 3
    wikidata_rows = []
    
    def create_batch(list, dimension):
        for i in range(0, len(list), dimension):
            yield list[i:i + dimension]
    
    for index_batch, batch in enumerate(create_batch(countries_list, BATCH_SIZE), start=1):
        nomi_lotto = ", ".join([p[2] for p in batch])
    
        qid_clean = []
        for p in batch:
            raw_qid = p[1]
            clean_qid = raw_qid.split("/")[-1].replace("wd:", "")
            qid_clean.append(f"wd:{clean_qid}")
    
        string_values = " ".join(qid_clean)
    
        # museums, libraries, archives, galleries
        query_batch = f"""
        SELECT ?country (COUNT(DISTINCT ?item) AS ?count) WHERE {{
          VALUES ?country {{ {string_values} }}
          VALUES ?type {{ wd:Q33506 wd:Q7075 wd:Q166118 wd:Q1007870 }}
    
          ?item wdt:P31 ?type ;
                wdt:P17 ?country .
        }}
        GROUP BY ?country
        """
    
        sparql.setQuery(query_batch)
    
        try:
            results = sparql.query().convert()
    
            count_temp = {}
            for row in results["results"]["bindings"]:
                qid = row["country"]["value"].split("/")[-1]
                count_temp[qid] = int(row["count"]["value"])
    
            for iso, qid, nome in batch:
                clean_qid = qid.split("/")[-1].replace("wd:", "")
                values = count_temp.get(clean_qid, 0)
                wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': values})
    
        except Exception as e:
            print(f"   Batch Error {index_batch}: {e}")
            for iso, qid, nome in batch:
                wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': 0})
    
        if index_batch * BATCH_SIZE < len(countries_list):
            time.sleep(1.5)
    
    df_wikidata = pd.DataFrame(wikidata_rows)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_wikidata.to_csv(csv_path, index=False)

df_wikidata


Loading Wikidata GLAM data from cache...


,iso_code,glam_count_wikidata
0,IT,14236
1,DE,9370
2,NL,1539
3,PT,554
4,ES,2813
5,FR,2640


#### Integrating the result
The four disparate tables (`df_countries`, `df_europeana`, `df_exp_filtered`, `df_wikidata`) are merged via an inner relational join on the `iso_code` primary key.
A new column is added with the **GLAM Partecipation Rate (%)**: 
  $$\text{Partecipation Rate} = \left(\frac{\text{europeana\_providers}}{\text{glam\_count\_wikidata}}\right) \times 100$$
**Display Formatting**: Numeric variables are cast to float/int and formatted using Pandas Styler (`{:,.0f}` for totals, `{:.2f}%` for rates) to ensure clean tabular presentation without modifying underlying float values.

In [10]:
# Data integration
df_final = df_countries.merge(df_europeana, on='iso_code') \
                    .merge(df_exp_filtered, on='iso_code') \
                    .merge(df_wikidata, on='iso_code') 

df_final['partecipation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

formatted_df = df_final.style.format({
    'europeana_total': '{:,.0f}',
    'partecipation_rate_glam': '{:.2f}%',
    'culture_expenditure_gpd_2022': '{:.2f}%'
})

formatted_df

,iso_code,name_en,wd_id,europeana_total,europeana_providers,culture_expenditure_gpd_2022,glam_count_wikidata,partecipation_rate_glam
0,IT,Italy,Q38,"1,832,376",158,0.90%,14236,1.11%
1,DE,Germany,Q183,"8,701,240",375,1.00%,9370,4.00%
2,NL,Netherlands,Q55,"9,204,845",104,1.10%,1539,6.76%
3,PT,Portugal,Q45,"139,858",40,0.90%,554,7.22%
4,ES,Spain,Q29,"6,581,724",275,1.20%,2813,9.78%
5,FR,France,Q142,"4,724,898",50,1.40%,2640,1.89%


---

#### Visualizing RsQ01
* **Chart Type**: Normalized 100% Stacked Bar Chart (`plotly.express.bar`).
* **Data Transformation**: The partecipation rate is melted into a tidy format representing two complementary percentage states: `Active on Europeana (%)` and `Offline GLAMs (%)`.
* **Visual Encoding**: Countries are sorted in descending order by partecipation rate. Bar text labels display percentages rounded to one decimal place (`textposition='outside'` for active segments, `hoverinfo='none'` to disable interactive tooltips for clean static rendering).

In [11]:
# Visualization Partecipation Rate Glam
df_support = df_final[['iso_code', 'partecipation_rate_glam']].copy()
df_support['Active on Europeana (%)'] = df_support['partecipation_rate_glam']
df_support['Offline GLAMs (%)'] = 100 - df_support['partecipation_rate_glam']

df_support = df_support.sort_values(by='partecipation_rate_glam', ascending=False)

df_tidy = df_support.melt(
    id_vars=['iso_code', 'partecipation_rate_glam'],
    value_vars=['Active on Europeana (%)', 'Offline GLAMs (%)'],
    var_name='Status',
    value_name='Percentage'
)

fig1 = px.bar(
    df_tidy,
    x='iso_code',
    y='Percentage',
    color='Status',
    title="The Partecipation Gap: distribution of active europeana providers",
    labels={
        'iso_code': 'Country',
        'Percentage': 'Share of Physical GLAMs (%)',
        'Status': 'Institutional Status'
    },
    color_discrete_map={
        'Active on Europeana (%)': '#5A5A5A',
        'Offline GLAMs (%)': '#AACAE0'
    }
)

fig1.update_traces(
    texttemplate='',
    hoverinfo='none'
)

df_active = df_tidy[df_tidy['Status'] == 'Active on Europeana (%)']

fig1.add_trace(
    go.Scatter(
        x=df_active['iso_code'],
        y=df_active['Percentage'],
        text=df_active['Percentage'].map('{:.1f}%'.format),
        mode='text',
        textposition='top center',
        textfont=dict(size=12, color='black'),
        showlegend=False,
        hoverinfo='none'
    )
)

fig1.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=600,
    width=800,
    barmode='stack',
    hovermode=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[0, 108]
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig1.show()

#### Visualizing RsQ02
* **Chart Type**: Scatter Plot with Ordinary Least Squares (OLS) Regression Line.
* **Variables**: X-axis encodes public cultural expenditure (% GDP); Y-axis encodes total Europeana digital objects.
* **Model Parameters**: An OLS trendline ($y = mx + q$) is fitted using `numpy.polyfit`. To test the hypothesis of zero digital output at zero public spending, a forced-origin regression ($q = 0$) is calculated where $m = \frac{\sum(xy)}{\sum(x^2)}$.

In [12]:
# Visualization Culture expenditure correlation
x_vals = df_final['culture_expenditure_gpd_2022'].values
y_vals = df_final['europeana_total'].values

m = np.sum(x_vals * y_vals) / np.sum(x_vals**2)
q = 0

x_line = np.linspace(0, x_vals.max() + 0.05, 100)
y_line = m * x_line

fig2 = px.scatter(
    df_final,
    x="culture_expenditure_gpd_2022",
    y="europeana_total",
    text="iso_code",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    title="Relationship between public funding and total shared items in Europeana",
    labels={
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects Shared on Europeana"
    },
    hover_data={
        "europeana_total": ":,.0f",
        "culture_expenditure_gpd_2022": False,
        "iso_code": False
    }
)

fig2.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name="Global Trendline (OLS)",
        line=dict(dash="dash", color="#FF4B4B", width=2.5),
        showlegend=False,
        hoverinfo="skip"
    )
)

fig2.update_layout(
    template="plotly_white",
    height=550,
    width=750,
    showlegend=False,
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[x_vals.min() - 0.08, x_vals.max() + 0.08]
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        tickformat=","
    )
)

fig2.update_traces(
    selector=dict(mode="markers+text"),
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)


fig2.show()

#### Visualizing RsQ03 
* **Chart Type**: Multivariate Bubble Chart (`plotly.express.scatter`).
* **Visual Encoding**: X-axis = GLAM Partecipation Rate (%); Y-axis = Public Cultural Expenditure (% GDP); Bubble Area (`size`) = Total Europeana Objects (`size_max=65`).


In [13]:
# Visualization relationship between investment, glam partecipation rate, europeana total item for each selected countries
fig3 = px.scatter(
    df_final,
    x="partecipation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    text="iso_code",
    hover_name="name_en",

    hover_data={
        "europeana_total": ":,",
        "partecipation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },

    title="European(a) Digital Heritage: relationship between investment, <br>glam partecipation rate and total shared items</br>",
    labels={
        "partecipation_rate_glam": "GLAM Partecipation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()



#### Updating Visualization RsQ03
**Sensitivity Override**: To test metric stability against crowdsourced semantic graph under-sampling, the Wikidata baseline for Portugal (`PT`) is replaced with official administrative census data manually collected from INE, recording the most recent numerical data of  museums, libraries and galleries (`portugal_glam_census.csv`, $N=3,395$). The X-axis spatial grid (`range=[-0.5, 12.5]`) is locked across both charts to visually isolate the mathematical impact of the denominator override.

In [14]:
# Update with data from portugal GLAM census
df_ine = pd.read_csv('data/csv/portugal_glam_census.csv')
pt_real_glam_total = df_ine['Value'].sum()
df_final.loc[df_final['iso_code'] == 'PT', 'glam_count_wikidata'] = pt_real_glam_total

df_final['partecipation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

fig3 = px.scatter(
    df_final,
    x="partecipation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    color_discrete_map=COUNTRY_COLORS_ISO,
    text="iso_code",
    hover_name="name_en",
    hover_data={
        "europeana_total": ":,",
        "partecipation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },
    title="Updated European(a) Digital Heritage: relationship between investment, <br> glam partecipation rate (updated) and total shared items </br>",
    labels={
        "partecipation_rate_glam": "GLAM Partecipation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()

---

## RQ02 Who are the providers contributing content to Europeana, and how are they distributed?

### RsQ04 Which are the providers contributing to Europeana?
The first subquestion identifies, for every country, the full set of institutions contributing to Europeana. We retrieve the complete list of distinct data providers together with their item counts via Europeana's API, establishing how many and which institutions each country actually relies on to build its digital presence.

### RsQ05 How much of each country's total item volume is concentrated in its top providers?
The second subquestion examines the distribution of contributions to country coverage in Europeana, assessing whether they are spread across many institutions with comparable levels of participation or concentrated in the hands of one or two dominant providers that account for the majority of the records.

### RsQ06 How are these providers geographically distributed across Europe?
The third subquestion continues the analysis of concentration by shifting the focus to the geographical level. By mapping the top 20 providers by item count for each country, we investigate where digitization activity is concentrated, assessing whether national contributions result from a geographically distributed effort involving both major institutions in large urban centers and smaller organizations in peripheral locations, or whether digitization activity remains confined to the largest cities. To produce this analysis, we resolve the city-level location of each provider using Europeana's Organization entity profiles, which include address and geospatial information for most organizations. When this information is unavailable, we supplement it with data from Wikidata or manual data collection.

### RsQ07 What types of providers contribute to Europeana?
This last step investigates what kind of institutions contribute to Europeana,
shifting the focus from the providers themselves to what they are actually contributing to Europeana. Each provider is classified into an institution-type category derived from the observed data rather than defined upfront. Classification draws primarily on the provider's Wikidata entity and its properties.

#### Querying the Europeana API for full provider lists and volume percentages

Data is collected via the Europeana Search REST API. The functions iteratively query each country to retrieve the full list of providers and their respective item volumes.

Based on these calculated volumes, the lists are then sorted and truncated to the top 100 providers per country. This optimizes the subsequent classification process while ensuring the relative weight of the national aggregates is accurately captured.

In [15]:
# Europeana Provider Data Collection
import re

TARGET_COUNTRIES = [
    "italy", "france", "germany", "spain", "netherlands", "portugal",
]
LANG_BY_COUNTRY = {
    "italy": "it", "france": "fr", "germany": "de",
    "spain": "es", "netherlands": "nl", "portugal": "pt",
}

API_KEY = EUROPEANA_API_KEY

def europeana_search(query="*", qf=None, facet=None, rows=0, **extra):
    params = {"wskey": API_KEY, "query": query, "rows": rows, **extra}
    if qf:
        params["qf"] = qf
    if facet:
        params["facet"] = facet
        params["profile"] = "facets"

    for attempt in range(5):
        resp = requests.get(EUROPEANA_SEARCH_URL, params=params)
        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        if resp.status_code != 200:
            print("URL:", resp.url)
            print("Status:", resp.status_code)
            print("Body:", resp.text[:1000])
            resp.raise_for_status()
        time.sleep(REQUEST_DELAY)
        return resp.json()

    raise RuntimeError(f"Failed after retries: {params}")


test = europeana_search(query="*", qf=["COUNTRY:italy"], rows=1)
print("success:", test.get("success"), "| totalResults:", test.get("totalResults"))

def get_total_items(country):
    result = europeana_search(query="*", qf=[f"COUNTRY:{country}"], rows=0)
    return {"country": country, "total_items": result.get("totalResults", 0)}


totals_df = pd.DataFrame([get_total_items(c) for c in TARGET_COUNTRIES])
display(totals_df)

def get_all_providers(country, page_size=200, max_pages=50):
    rows = []
    offset = 0
    for _ in range(max_pages):
        result = europeana_search(
            query="*",
            qf=[f"COUNTRY:{country}"],
            facet="DATA_PROVIDER",
            rows=0,
            **{"f.DATA_PROVIDER.facet.limit": page_size, "f.DATA_PROVIDER.facet.offset": offset},
        )
        facets = result.get("facets", [])
        page_rows = [
            {"country": country, "provider": f["label"], "count": f["count"]}
            for facet in facets
            for f in facet.get("fields", [])
        ]
        rows.extend(page_rows)
        if len(page_rows) < page_size:
            break
        offset += page_size
    return rows


providers_by_country = {}
target_folder = DATA_DIR / "csv" / "providers_data"
target_folder.mkdir(parents=True, exist_ok=True)

for country in TARGET_COUNTRIES:
    rows = get_all_providers(country)
    country_df = pd.DataFrame(rows)
    country_df.to_csv(target_folder / f"providers_{country}.csv", index=False)
    providers_by_country[country] = country_df
    print(f"{country}: {len(country_df)} distinct providers -> providers_{country}.csv")

providers_df = pd.concat(providers_by_country.values(), ignore_index=True)


success: True | totalResults: 1832376


,country,total_items
0,italy,1832376
1,france,4724898
2,germany,8701240
3,spain,6581724
4,netherlands,9204845
5,portugal,139858


italy: 158 distinct providers -> providers_italy.csv
france: 50 distinct providers -> providers_france.csv
germany: 375 distinct providers -> providers_germany.csv
spain: 275 distinct providers -> providers_spain.csv
netherlands: 104 distinct providers -> providers_netherlands.csv
portugal: 40 distinct providers -> providers_portugal.csv


In [16]:
# what % of total item volume the top N providers represent - how large N needs to be.

def coverage_at_n(country_df, n):
    sorted_df = country_df.sort_values("count", ascending=False)
    return sorted_df["count"].head(n).sum() / sorted_df["count"].sum()

for country, df in providers_by_country.items():
    print(f"{country}: top 20 -> {coverage_at_n(df, 20):.1%}, top 50 -> {coverage_at_n(df, 50):.1%}, "
          f"top 100 -> {coverage_at_n(df, 100):.1%} (of {len(df)} total providers)")

italy: top 20 -> 90.2%, top 50 -> 98.4%, top 100 -> 99.9% (of 158 total providers)
france: top 20 -> 99.7%, top 50 -> 100.0%, top 100 -> 100.0% (of 50 total providers)
germany: top 20 -> 79.6%, top 50 -> 92.2%, top 100 -> 97.7% (of 375 total providers)
spain: top 20 -> 76.7%, top 50 -> 92.3%, top 100 -> 98.4% (of 275 total providers)
netherlands: top 20 -> 89.8%, top 50 -> 99.5%, top 100 -> 100.0% (of 104 total providers)
portugal: top 20 -> 99.7%, top 50 -> 100.0%, top 100 -> 100.0% (of 40 total providers)


In [17]:
# Aggregating Top Providers per Country
# cut each country's provider list down to its top 100 by item count
TOP_N_PER_COUNTRY = 100

top_providers_rows = []
for country, df in providers_by_country.items():
    top = df.sort_values("count", ascending=False).head(TOP_N_PER_COUNTRY)
    top_providers_rows.append(top)

top_providers_df = pd.concat(top_providers_rows, ignore_index=True)
distinct_providers = top_providers_df[["provider", "country"]].drop_duplicates()
print(f"{len(top_providers_df)} (country, provider) rows / {len(distinct_providers)} distinct providers selected")


490 (country, provider) rows / 490 distinct providers selected


#### Calculating provider concentration of the top 5 providers

For each country, we measure the extent to which its total item volume is concentrated among its 5 largest providers. Specifically, we identify the single largest provider (`top_provider`) and calculate two indicators: the share of the country's total item count contributed by that provider alone (`pct_top1_provider`) and the combined share contributed by the top five providers (`pct_top5_providers`). Countries are then ranked from the highest to the lowest level of provider concentration.

In [18]:
def provider_concentration(providers_dir=None, totals_df=totals_df, top_n=5):
    providers_dir = providers_dir or (DATA_DIR / "csv" / "providers_data")
    totals = totals_df.set_index("country")["total_items"].to_dict()

    rows = []
    for country in TARGET_COUNTRIES:
        csv_path = providers_dir / f"providers_{country}.csv"
        country_df = pd.read_csv(csv_path)
        if country_df.empty:
            continue

        total = totals.get(country)
        sorted_counts = country_df["count"].sort_values(ascending=False).tolist()
        top_provider_name = country_df.loc[country_df["count"].idxmax(), "provider"]

        top1_share = sorted_counts[0] / total
        top_n_actual_share = sum(sorted_counts[:top_n]) / total

        rows.append({
            "country": country,
            "num_providers": len(country_df),
            "top_provider": top_provider_name,
            "pct_top1_provider": round(100 * top1_share, 1),
            f"pct_top{top_n}_providers": round(100 * top_n_actual_share, 1),
        })

    return pd.DataFrame(rows).sort_values("pct_top1_provider", ascending=False)


concentration_df = provider_concentration(top_n=5)
concentration_df

,country,num_providers,top_provider,pct_top1_provider,pct_top5_providers
1,france,50,National Library of France,63.5,91.3
4,netherlands,104,Naturalis Biodiversity Center,50.0,73.2
5,portugal,40,Institute for Tropical Scientific Research,47.0,89.8
3,spain,275,Virtual Library of Historical Press,26.4,49.5
2,germany,375,Bavarian State Library,25.0,56.8
0,italy,158,Cinecittà - Luce,24.2,58.1


#### Querying Europeana and Wikidata for geo location

To map where digitization activity concentrates within each country, we resolve a city-level location for each country's top 20 providers by item count, using two sources queried in sequence.

* **Europeana Entity API Lookup**: Each provider name is searched via the `/entity/suggest` endpoint to retrieve its Organization entity record. Where present, the `hasAddress.hasGeo` property supplies direct latitude/longitude and city (`locality`) values, curated by Europeana itself.
* **Wikidata Fallback**: For providers without a usable Europeana geo, resolution falls back to Wikidata via two paths — first using the Wikidata URI already cross-referenced in the Europeana entity's `sameAs` field (no name-matching required), then, if absent, via a name-based search against Wikidata's REST API. Coordinates (`wdt:P625`) and city (`wdt:P131`) are retrieved via the SPARQL endpoint.
* **Manual Resolution**: Providers unresolved through either automated path are geolocated by hand.

Each resolved record retains a `source` tag (`europeana_entity`, `wikidata_via_europeana_link`, `wikidata_name_search`, or `manual`) for methodological transparency.

In [19]:
def top_n_providers(providers_dir=None, n=20):
    providers_dir = providers_dir or (DATA_DIR / "csv" / "providers_data")

    rows = []
    for country in TARGET_COUNTRIES:
        csv_path = providers_dir / f"providers_{country}.csv"
        country_df = pd.read_csv(csv_path)
        top_providers = country_df.sort_values("count", ascending=False).head(n)
        for _, row in top_providers.iterrows():
            rows.append({"country": country, "provider": row["provider"], "count": row["count"]})

    return pd.DataFrame(rows)


top20_df = top_n_providers()
top20_df

,country,provider,count
0,italy,Cinecittà - Luce,442536
1,italy,Historical Archive of the Presidency of the Re...,212160
2,italy,National Central Library of Rome,200231
3,italy,Internet Culturale,109101
4,italy,"Department of Life Sciences, University of Tri...",101323
...,...,...,...
115,portugal,Lisbon Newspaper Library,468
116,portugal,April 25th Documentation Center of Coimbra Uni...,337
117,portugal,The Commission for Citizenship and Gender Equa...,311
118,portugal,Division of Archives and Library of the Minist...,158


In [20]:
import requests
import time
import json

# STEP 1 — Look up each provider in Europeana's own Organization entity system

def search_organization_entity(name):
    url = "https://api.europeana.eu/entity/suggest"
    params = {"wskey": EUROPEANA_API_KEY, "text": name, "type": "organization"}
    r = requests.get(url, params=params, timeout=10)
    if r.status_code != 200:
        return None
    items = r.json().get("items", [])
    return items[0]["id"] if items else None  # take the top match


def fetch_organization_entity(entity_id):
    entity_num = entity_id.rstrip("/").split("/")[-1]
    url = f"https://api.europeana.eu/entity/organization/{entity_num}"
    params = {"wskey": EUROPEANA_API_KEY}
    r = requests.get(url, params=params, timeout=10)
    if r.status_code != 200:
        return None
    return r.json()


def get_provider_geo(name):
    entity_id = search_organization_entity(name)
    if not entity_id:
        return {"matched": False}

    entity = fetch_organization_entity(entity_id)
    if not entity:
        return {"matched": True, "entity_id": entity_id, "has_geo": False}

    address = entity.get("hasAddress", {})
    geo = address.get("hasGeo", {})
    wikidata_uri = next((s for s in entity.get("sameAs", []) if "wikidata.org" in s), None)

    return {
        "matched": True,
        "entity_id": entity_id,
        "has_geo": bool(geo),
        "locality": address.get("locality"),
        "latitude": float(geo["lat"]) if geo.get("lat") else None,
        "longitude": float(geo["long"]) if geo.get("long") else None,
        "wikidata_uri": wikidata_uri,
    }


def enrich_providers_with_geo(top20_df):
    results = []
    for _, row in top20_df.iterrows():
        geo_info = get_provider_geo(row["provider"])
        results.append({
            "country": row["country"],
            "provider": row["provider"],
            "count": row["count"],
            **geo_info,
        })
        time.sleep(0.2)
    return pd.DataFrame(results)


geo_df = enrich_providers_with_geo(top20_df)

geo_dir = DATA_DIR / "csv"
geo_dir.mkdir(parents=True, exist_ok=True)
geo_csv_path = geo_dir / "geo_df.csv"

if geo_csv_path.exists():
    geo_df = pd.read_csv(geo_csv_path)
    print(f"Loaded cached geo_df from {geo_csv_path} ({len(geo_df)} rows)")
else:
    geo_df = enrich_providers_with_geo(top20_df)
    geo_df.to_csv(geo_csv_path, index=False)
    print(f"Fetched and saved geo_df to {geo_csv_path}")

geo_df

Loaded cached geo_df from data/csv/geo_df.csv (120 rows)


,country,provider,count,matched,entity_id,has_geo,locality,latitude,longitude,wikidata_uri
0,italy,Cinecittà - Luce,442536,False,NaN,NaN,NaN,NaN,NaN,NaN
1,italy,Historical Archive of the Presidency of the Re...,212160,True,http://data.europeana.eu/organization/1117,False,Rome,NaN,NaN,NaN
2,italy,National Central Library of Rome,200231,True,http://data.europeana.eu/organization/4044,False,Rome,NaN,NaN,http://www.wikidata.org/entity/Q6382871
3,italy,Internet Culturale,109101,True,http://data.europeana.eu/organization/3262,True,Roma,41.906175,12.508847,NaN
4,italy,"Department of Life Sciences, University of Tri...",101323,True,http://data.europeana.eu/organization/1752,True,Trieste,45.660689,13.795708,http://www.wikidata.org/entity/Q75266481
...,...,...,...,...,...,...,...,...,...,...
115,portugal,Lisbon Newspaper Library,468,True,http://data.europeana.eu/organization/3494,True,Lisboa,38.754827,-9.172655,NaN
116,portugal,April 25th Documentation Center of Coimbra Uni...,337,True,http://data.europeana.eu/organization/3996,True,Coimbra,40.213878,-8.431251,NaN
117,portugal,The Commission for Citizenship and Gender Equa...,311,True,http://data.europeana.eu/organization/3834,True,Lisbon,38.713244,-9.166362,NaN
118,portugal,Division of Archives and Library of the Minist...,158,True,http://data.europeana.eu/organization/2911,True,Lisbon,38.707325,-9.170275,NaN


In [21]:
no_geo_but_matched = geo_df[(geo_df["matched"]) & (~geo_df["has_geo"].fillna(False))]
unmatched = geo_df[~geo_df["matched"]]

print(f"Matched with geo: {geo_df['has_geo'].sum()}")
print(f"Matched, no geo: {len(no_geo_but_matched)}")
print(f"Unmatched entirely: {len(unmatched)}")

Matched with geo: 74
Matched, no geo: 25
Unmatched entirely: 21


/var/folders/0x/pqmm6nvs7zn3rpwjrpfsd8pw0000gn/T/ipykernel_8263/4154389819.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  no_geo_but_matched = geo_df[(geo_df["matched"]) & (~geo_df["has_geo"].fillna(False))]


In [22]:
from SPARQLWrapper import SPARQLWrapper, JSON
import re
import pandas as pd

HEADERS = {
    "User-Agent": "cultural-heritage-research/0.1 (university project; contact: your_email@studio.unibo.it)"}

def wikidata_search(name, limit=1):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "format": "json",
        "limit": limit,
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return r.json().get("search", [])


def match_providers_to_wikidata(top20_df):
    matches = []
    for _, row in top20_df.iterrows():
        results = wikidata_search(row["provider"])
        if results:
            match = results[0]
            matches.append({
                "country": row["country"],
                "provider": row["provider"],
                "count": row["count"],
                "wikidata_id": match["id"],
                "wikidata_label": match.get("label"),
                "wikidata_description": match.get("description"),
            })
        else:
            matches.append({
                "country": row["country"],
                "provider": row["provider"],
                "count": row["count"],
                "wikidata_id": None,
                "wikidata_label": None,
                "wikidata_description": None,
            })
        time.sleep(0.2)  # be polite to the API
    return pd.DataFrame(matches)

def parse_point(coord_str):
    # Wikidata returns WKT format: "Point(lon lat)"
    match = re.match(r"Point\(([-\d.]+) ([-\d.]+)\)", coord_str)
    if match:
        lon, lat = match.groups()
        return float(lat), float(lon)
    return None, None

def fetch_coordinates(qids):
    """Given a list of Wikidata QIDs, return coordinates (P625) + city (P131)."""
    if not qids:
        return pd.DataFrame(columns=["wikidata_id", "wikidata_label", "city", "latitude", "longitude"])

    endpoint = "https://query.wikidata.org/sparql"
    sparql = SPARQLWrapper(endpoint, agent=HEADERS["User-Agent"])

    values_clause = " ".join(f"wd:{qid}" for qid in qids)
    query = f"""
    SELECT ?item ?itemLabel ?coord ?cityLabel WHERE {{
      VALUES ?item {{ {values_clause} }}
      ?item wdt:P625 ?coord .
      OPTIONAL {{ ?item wdt:P131 ?city . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    rows = []
    for r in results["results"]["bindings"]:
        lat, lon = parse_point(r["coord"]["value"])
        rows.append({
            "wikidata_id": r["item"]["value"].split("/")[-1],
            "wikidata_label": r["itemLabel"]["value"],
            "city": r.get("cityLabel", {}).get("value"),
            "latitude": lat,
            "longitude": lon,
        })
    return pd.DataFrame(rows)

def extract_qid(wikidata_uri):
    return wikidata_uri.rstrip("/").split("/")[-1]

In [23]:
# STEP 2 — Wikidata fallback #1: use the Wikidata URI Europeana already cross-referenced 
fallback_qids_direct = no_geo_but_matched["wikidata_uri"].dropna().apply(extract_qid).tolist()
fallback_coords_direct = fetch_coordinates(fallback_qids_direct)

fallback_coords_direct

HTTPError: HTTP Error 429: Too Many Requests

In [ ]:
# QIDs that successfully resolved to coordinates via the direct Wikidata link
resolved_qids_step5 = set(fallback_coords_direct["wikidata_id"]) if len(fallback_coords_direct) else set()

still_unresolved = pd.concat([
    unmatched[["country", "provider", "count"]],
    no_geo_but_matched[~no_geo_but_matched["wikidata_uri"].apply(
        lambda u: pd.notna(u) and extract_qid(u) in resolved_qids_step5
    )][["country", "provider", "count"]],
], ignore_index=True)

print(f"Still unresolved before name search: {len(still_unresolved)}")

# STEP 3 - Wikidata Fallback #2: fuzzy name-based search against Wikidata for the remaining providers.
matched_fallback_df = match_providers_to_wikidata(still_unresolved)
print(matched_fallback_df["wikidata_id"].notna().sum(), "of", len(matched_fallback_df), "matched via name search")

# Fetch coordinates
fallback_qids_name = matched_fallback_df.loc[matched_fallback_df["wikidata_id"].notna(), "wikidata_id"].tolist()
fallback_coords_name = fetch_coordinates(fallback_qids_name)
fallback_coords_name

Still unresolved before name search: 43
14 of 43 matched via name search


,wikidata_id,wikidata_label,city,latitude,longitude
0,Q163255,Botanic Garden and Botanical Museum Berlin,Steglitz-Zehlendorf,52.455000,13.303600
1,Q655507,Deutsche Fotothek,Dresden,51.027810,13.736740
2,Q821048,Berlin-Brandenburg Economic Archive,Berlin,52.584017,13.316706
3,Q1092493,Cineteca di Bologna,Bologna,44.498866,11.336828
4,Q1954331,Museon-Omniversum,The Hague,52.088778,4.281000
5,Q1954426,Museum Catharijneconvent,Utrecht,52.087222,5.124167
6,Q3052794,Calouste Gulbenkian Foundation,Lisbon,38.737220,-9.154170
7,Q3378907,Philharmonie de Paris,Paris,48.891566,2.394070
8,Q3639582,Biblioteca Europea di Informazione e Cultura,Milan,45.472601,9.188543
9,Q41328625,Girona City Council,None,41.983138,2.824800


In [ ]:
# Consolidate evrything into one final table

# Providers with direct Europeana geo
europeana_geo = geo_df[geo_df["has_geo"].fillna(False)][
    ["country", "provider", "count", "locality", "latitude", "longitude"]
].rename(columns={"locality": "city"})
europeana_geo["source"] = "europeana_entity"

# Providers resolved via Europeana's own Wikidata cross-reference
wikidata_direct = no_geo_but_matched.drop(columns=["latitude", "longitude"]).copy()
wikidata_direct["qid"] = wikidata_direct["wikidata_uri"].apply(
    lambda u: extract_qid(u) if pd.notna(u) else None
)
wikidata_direct = wikidata_direct.merge(
    fallback_coords_direct, left_on="qid", right_on="wikidata_id", how="inner"
)[["country", "provider", "count", "city", "latitude", "longitude"]]
wikidata_direct["source"] = "wikidata_via_europeana_link"

# Providers resolved via name-based Wikidata search
wikidata_named = matched_fallback_df.merge(
    fallback_coords_name, on="wikidata_id", how="inner"
)[["country", "provider", "count", "city", "latitude", "longitude"]]
wikidata_named["source"] = "wikidata_name_search"

provider_geo_final = pd.concat([europeana_geo, wikidata_direct, wikidata_named], ignore_index=True)
print(f"Total resolved: {len(provider_geo_final)} / {len(top20_df)}")
provider_geo_final

Total resolved: 88 / 120


/var/folders/0x/pqmm6nvs7zn3rpwjrpfsd8pw0000gn/T/ipykernel_5526/554173251.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  europeana_geo = geo_df[geo_df["has_geo"].fillna(False)][


,country,provider,count,city,latitude,longitude,source
0,italy,Internet Culturale,109101,Roma,41.906175,12.508847,europeana_entity
1,italy,"Department of Life Sciences, University of Tri...",101323,Trieste,45.660689,13.795708,europeana_entity
2,italy,Epigraphic Database Roma,86827,Rome,41.903763,12.514438,europeana_entity
3,italy,Rossimoda Shoe Museum,13489,Vigonza,45.421946,11.976820,europeana_entity
4,france,National Library of France,2999310,Paris,48.833584,2.375766,europeana_entity
...,...,...,...,...,...,...,...
83,netherlands,Museon-Omniversum,93077,The Hague,52.088778,4.281000,wikidata_name_search
84,netherlands,St. Catherine's Convent Museum,60795,Utrecht,52.087222,5.124167,wikidata_name_search
85,italy,European Library of Information and Culture,35475,Milan,45.472601,9.188543,wikidata_name_search
86,italy,Braidense National Library,21526,Milan,45.471947,9.187837,wikidata_name_search


#### Manual collection of geo location for providers left unresolved

For the 32 providers that remained unresolved after both automated paths, coordinates were collected manually.

During this review, one provider ("Ministry of Culture," France) was found to have matched an incorrect entity earlier in the automated lookup, due to name ambiguity across countries — the query had returned a Romanian institution of the same generic name. This was corrected, and given the risk of similarly generic institution names, all resolved (country, city) pairs across the full dataset were spot-checked for
geographic plausibility.

In [ ]:
resolved_providers = set(provider_geo_final["provider"])
truly_unresolved = top20_df[~top20_df["provider"].isin(resolved_providers)]
truly_unresolved.sort_values("count", ascending=False)

,country,provider,count
60,spain,Virtual Library of Historical Press,1735320
42,germany,State Archives of Baden-Württemberg,827603
0,italy,Cinecittà - Luce,442536
64,spain,Galiciana. Arquivo Dixital de Galicia,234326
1,italy,Historical Archive of the Presidency of the Re...,212160
46,germany,Library of the Friedrich Ebert Foundation,201518
67,spain,Digital Library of Andalusia,145620
68,spain,Maresía: Prensa digitalizada y Patrimonio docu...,141919
50,germany,Archives of Social Democracy,133032
51,germany,State and University Library Hamburg Carl von ...,131101


In [ ]:
# Manual lookup
manual_geo = pd.DataFrame([
    {"country": "Spain", "provider": "Virtual Library of Historical Press", "count": 1735320,
     "city": "Madrid", "latitude": 40.4206699298513, "longitude":  -3.696534590397076, "source": "manual"},
    {"country": "Germany", "provider": "State Archives of Baden-Württemberg", "count": 827603,
     "city": "Stuttgart", "latitude": 48.848397483733265, "longitude":  9.185433331037915, "source": "manual"},
    {"country": "Italy", "provider": "Cinecittà - Luce", "count": 442536,
     "city": "Rome", "latitude": 41.849810560200865, "longitude": 12.574519080840693, "source": "manual"},
    {"country": "Spain", "provider": "Galiciana. Arquivo Dixital de Galicia", "count": 234326,
     "city": "Santiago de Compostela", "latitude": 42.94991406654687, "longitude":  -8.551490373538044, "source": "manual"},
    {"country": "Italy", "provider": "Historical Archive of the Presidency of the Republic", "count": 212160,
     "city": "Rome", "latitude": 41.90046503888887, "longitude": 12.488921680843333, "source": "manual"},
    {"country": "Germany", "provider": "Library of the Friedrich Ebert Foundation", "count": 201518,
     "city": "Bonn", "latitude": 50.70257995350261, "longitude":  7.135440740863708, "source": "manual"},
    {"country": "Spain", "provider": "Digital Library of Andalusia", "count": 145620,
     "city": "Granada", "latitude": 37.18282776503285, "longitude": -3.6056596093053677, "source": "manual"},
    {"country": "Spain", "provider": "Maresía: Prensa digitalizada y Patrimonio documental", "count": 141919,
     "city": "San Cristóbal de La Laguna", "latitude": 28.46926099886812, "longitude": -16.30470874858524, "source": "manual"},
    {"country": "Germany", "provider": "Archives of Social Democracy", "count": 133032,
     "city": "Bonn", "latitude": 50.702368944477904, "longitude":7.134824767850492, "source": "manual"},
    {"country": "Germany", "provider": "State and University Library Hamburg Carl von Ossietzky", "count": 131101,
     "city": "Hamburg", "latitude": 53.56490744018377, "longitude": 9.985154223852446, "source": "manual"},
    {"country": "Spain", "provider": "Canary Islands Historical Photography Archive", "count": 121890,
     "city": "Las Palmas de Gran Canaria", "latitude": 28.10709632881129, "longitude": -15.417504652154406, "source": "manual"},
    {"country": "Germany", "provider": "Teßmann Library", "count": 115179,
     "city": "Italy", "latitude": 46.50323665221172, "longitude": 11.342929852259413, "source": "manual"},
    {"country": "Spain", "provider": "Centro de Estudios de Castilla - La Mancha", "count": 94223,
     "city": "Ciudad Real", "latitude": 38.993031000616305, "longitude": -3.9196156769745314, "source": "manual"},
    {"country": "Spain", "provider": "Foundation Virtual Library Miguel de Cervantes", "count": 70971,
     "city": "Alicante", "latitude": 38.38507646054709, "longitude": -0.5139165229076937, "source": "manual"},
    {"country": "Spain", "provider": "Digital Library of Madrid", "count": 69472,
     "city": "Madrid", "latitude": 40.39987943495303, "longitude": -3.690579871358554, "source": "manual"},
    {"country": "Italy", "provider": "Central Institute for the Union Catalogue of Italian Libraries", "count": 66672,
     "city": "Rome", "latitude": 41.90631839791796, "longitude": 12.50879375200811, "source": "manual"},
    {"country": "Netherlands", "provider": "IMSLP/Petrucci Music Library", "count": 61552,
     "city": None, "latitude": None, "longitude": None, "source": "manual"},
    {"country": "Italy", "provider": "Central Museum of the Risorgimento", "count": 53211,
     "city": "Rome", "latitude": 41.89413575264878, "longitude": 12.483798667349626, "source": "manual"},
    {"country": "France", "provider": "Palais Galliera - Musée de la Mode de la Ville de Paris", "count": 44495,
     "city": "Paris", "latitude": 48.86608033629242, "longitude": 2.2965618523973155, "source": "manual"},
    {"country": "Italy", "provider": "Experimental Cinematography Center", "count": 40538,
     "city": "Rome", "latitude": 41.851058155342265, "longitude": 12.569479552005237, "source": "manual"},
    {"country": "Italy", "provider": "Epigraphic Dabatase Bari", "count": 40283,
     "city": "Bari", "latitude": 41.12112736784277, "longitude": 16.868604802773277, "source": "manual"},
    {"country": "Italy", "provider": "Marciana National Library", "count": 29597,
     "city": "Venice", "latitude": 45.43353428986529, "longitude": 12.339422752198944, "source": "manual"},
    {"country": "Italy", "provider": "Turin Gallery for Modern and Contemporary Art", "count": 29395,
     "city": "Turin", "latitude": 45.065010771493675, "longitude": 7.66921379635602, "source": "manual"},
    {"country": "France", "provider": "National and University Library of Strasbourg", "count": 21320,
     "city": "Strasbourg", "latitude": 48.5872587063354, "longitude": 7.755901183065034, "source": "manual"},
    {"country": "Italy", "provider": "Library of the S. Pietro a Majella Conservatory", "count": 20154,
     "city": "Naples", "latitude": 40.849630767959596, "longitude": 14.252446682637897, "source": "manual"},
    {"country": "Italy", "provider": "Provincial Library Magna Capitana", "count": 13065,
     "city": "Foggia", "latitude": 41.45671868731526, "longitude": 15.558523953833472, "source": "manual"},
    {"country": "Italy", "provider": "Estense University Library", "count": 12372,
     "city": "Modena", "latitude": 44.64842579381791, "longitude": 10.920992367497412, "source": "manual"},
    {"country": "France", "provider": "Rhône-Alpes Laboratory for Historical Research", "count": 10046,
     "city": "Lyon", "latitude": 45.7334134679147, "longitude": 4.833417509886817, "source": "manual"},
    {"country": "Portugal", "provider": "MUDE – Museu do Design", "count": 1927,
     "city": "Lisbon", "latitude": 38.70924101252115, "longitude":  -9.136938317468571, "source": "manual"},
    {"country": "Portugal", "provider": "Fernando Pessoa's House", "count": 1210,
     "city": "Lisbon", "latitude": 38.716820396673036, "longitude": -9.162572105823484, "source": "manual"},
    {"country": "Portugal", "provider": "Lisbon's Film & Theatre School", "count": 774,
     "city": "Amadora", "latitude": 38.77728867945194, "longitude": -9.234926049524557, "source": "manual"},
    {"country": "Portugal", "provider": "Cinemateca Portuguesa - Museu do cinema", "count": 654,
     "city": "Lisbon", "latitude": 38.721224635535016, "longitude": -9.14865910212588, "source": "manual"},
])


provider_geo_final = pd.concat(
    [europeana_geo, wikidata_direct, wikidata_named, manual_geo],
    ignore_index=True
)

# Correct France 'Ministry of Culture' (automated lookup returned wrong country)
correct_coords = {"city": "Paris", "latitude": 48.86255840307406, "longitude": 2.3388283365026106}  # French Ministry of Culture, Paris 

mask = (provider_geo_final["provider"] == "Ministry of Culture") & \
       (provider_geo_final["country"].str.lower() == "france")

provider_geo_final.loc[mask, ["city", "latitude", "longitude"]] = list(correct_coords.values())
provider_geo_final.loc[mask, "source"] = "manual correction"

geo_dir = DATA_DIR / "csv"
geo_dir.mkdir(parents=True, exist_ok=True)
provider_geo_final_path = geo_dir / "provider_geo_final.csv"

provider_geo_final.to_csv(provider_geo_final_path, index=False)
print(f"Saved provider_geo_final to {provider_geo_final_path} ({len(provider_geo_final)} rows)")

# Normalize to lowercase country column
provider_geo_final["country"] = provider_geo_final["country"].str.lower()
provider_geo_final


Saved provider_geo_final to data/csv/provider_geo_final.csv (120 rows)


,country,provider,count,city,latitude,longitude,source
0,italy,Internet Culturale,109101,Roma,41.906175,12.508847,europeana_entity
1,italy,"Department of Life Sciences, University of Tri...",101323,Trieste,45.660689,13.795708,europeana_entity
2,italy,Epigraphic Database Roma,86827,Rome,41.903763,12.514438,europeana_entity
3,italy,Rossimoda Shoe Museum,13489,Vigonza,45.421946,11.976820,europeana_entity
4,france,National Library of France,2999310,Paris,48.833584,2.375766,europeana_entity
...,...,...,...,...,...,...,...
115,france,Rhône-Alpes Laboratory for Historical Research,10046,Lyon,45.733413,4.833418,manual
116,portugal,MUDE – Museu do Design,1927,Lisbon,38.709241,-9.136938,manual
117,portugal,Fernando Pessoa's House,1210,Lisbon,38.716820,-9.162572,manual
118,portugal,Lisbon's Film & Theatre School,774,Amadora,38.777289,-9.234926,manual


#### Querying the remote Wikidata APIs for provider metadata

To later classify the providers into specific typologies, both the Wikidata REST API and the SPARQL endpoint (`https://query.wikidata.org/sparql`) are queried.

* **REST API Entity Resolution**: The provider's name is searched (first in English, falling back to local language) to retrieve the entity's direct description.
* **SPARQL Endpoint Properties**: The endpoint is queried to extract two specific structural properties:
  * `wdt:P31` (instance of): Indicates the entity's class (e.g., museum, national library).
  * `wdt:P101` (field of work): Defines the entity's specialization (e.g., archaeology, broadcasting).
* **Batch Processing**: Provider QIDs are batched in groups of 50 (`batch_size=50`) to optimize query execution and prevent endpoint timeouts.
* **Output**: Returns the raw text descriptions, `instance_of`, and `field_of_work` properties, which form the basis for the subsequent classification.

In [ ]:
# Wikidata Metadata Collection
WD_SEARCH_URL = "https://www.wikidata.org/w/api.php"
WD_HEADERS = {
    "User-Agent": "InfoVis-course-project/0.1 (student project; contact: your.email@example.com)"
}
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"


def search_wikidata_rest(name, lang="en", max_retries=3):
    params = {
        "action": "wbsearchentities", "search": name, "language": lang,
        "format": "json", "limit": 1, "type": "item",
    }
    for attempt in range(max_retries):
        try:
            resp = requests.get(WD_SEARCH_URL, params=params, headers=WD_HEADERS, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"network error searching '{name}': {e}")
            time.sleep(2 ** attempt)
            continue
        if resp.status_code == 200:
            results = resp.json().get("search", [])
            if not results:
                return {"qid": None, "description": None, "desc_lang": None, "status": "no_match"}
            top = results[0]
            display_desc = top.get("display", {}).get("description", {})
            description = display_desc.get("value") or top.get("description")
            actual_lang = display_desc.get("language")
            return {"qid": top["id"], "description": description,
                    "desc_lang": actual_lang if description else None, "status": "ok"}
        if resp.status_code in (429, 502, 503):
            time.sleep(2 ** attempt)
            continue
        return {"qid": None, "description": None, "desc_lang": None, "status": f"http_{resp.status_code}"}
    return {"qid": None, "description": None, "desc_lang": None, "status": "retries_exhausted"}


def resolve_provider_wikidata(name, country):
    result = search_wikidata_rest(name, lang="en")
    if result["status"] == "ok" and result["description"]:
        result["resolution_lang"] = "en"
        return result
    country_lang = LANG_BY_COUNTRY.get(country, "en")
    local_result = search_wikidata_rest(name, lang=country_lang)
    if local_result["status"] == "ok" and (local_result["description"] or not result["qid"]):
        local_result["resolution_lang"] = country_lang
        return local_result
    if result["status"] == "ok":
        result["resolution_lang"] = "en"
        return result
    local_result["resolution_lang"] = country_lang
    return local_result


WD_RESOLUTION_FILE = DATA_DIR / "json" / "wikidata_resolution.json"
if WD_RESOLUTION_FILE.exists():
    with open(WD_RESOLUTION_FILE, "r", encoding="utf-8") as f:
        wikidata_resolution = json.load(f)
    print("Loaded wikidata resolution from cache.")
else:
    wikidata_resolution = {}
    for _, row in distinct_providers.iterrows():
        name, country = row["provider"], row["country"]
        result = resolve_provider_wikidata(name, country)
        wikidata_resolution[name] = result
        time.sleep(0.3)
    with open(WD_RESOLUTION_FILE, "w", encoding="utf-8") as f:
        json.dump(wikidata_resolution, f, indent=2, ensure_ascii=False)
    print("Saved wikidata resolution to cache.")
wd_resolved = sum(1 for r in wikidata_resolution.values() if r["status"] == "ok")
print(f"{wd_resolved} / {len(wikidata_resolution)} resolved via Wikidata")

def fetch_institution_info(qids, batch_size=50):
    results = {}
    qids = [q for q in qids if q]
    for i in range(0, len(qids), batch_size):
        batch = qids[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in batch)
        query = f"""
        SELECT ?item ?instanceOfLabel ?fieldOfWorkLabel WHERE {{
          VALUES ?item {{ {values} }}
          OPTIONAL {{ ?item wdt:P31 ?instanceOf . ?instanceOf rdfs:label ?instanceOfLabel . FILTER(LANG(?instanceOfLabel) = "en") }}
          OPTIONAL {{ ?item wdt:P101 ?fieldOfWork . ?fieldOfWork rdfs:label ?fieldOfWorkLabel . FILTER(LANG(?fieldOfWorkLabel) = "en") }}
        }}
        """
        resp = requests.get(WIKIDATA_SPARQL, params={"query": query, "format": "json"}, headers=WD_HEADERS)
        if resp.status_code != 200:
            continue
        for row in resp.json()["results"]["bindings"]:
            qid = row["item"]["value"].rsplit("/", 1)[-1]
            entry = results.setdefault(qid, {"instance_of": [], "field_of_work": []})
            if "instanceOfLabel" in row:
                entry["instance_of"].append(row["instanceOfLabel"]["value"])
            if "fieldOfWorkLabel" in row:
                entry["field_of_work"].append(row["fieldOfWorkLabel"]["value"])
        time.sleep(0.5)
    return results


SPARQL_INFO_FILE = DATA_DIR / "json" / "sparql_info.json"
if SPARQL_INFO_FILE.exists():
    with open(SPARQL_INFO_FILE, "r", encoding="utf-8") as f:
        sparql_info = json.load(f)
    print("Loaded SPARQL info from cache.")
else:
    qid_list = [info["qid"] for info in wikidata_resolution.values() if info.get("qid")]
    sparql_info = fetch_institution_info(qid_list)
    with open(SPARQL_INFO_FILE, "w", encoding="utf-8") as f:
        json.dump(sparql_info, f, indent=2, ensure_ascii=False)
    print("Saved SPARQL info to cache.")
qid_list = [info["qid"] for info in wikidata_resolution.values() if info.get("qid")]
print(f"instance_of/field_of_work found for {len(sparql_info)} / {len(qid_list)} QIDs")

Loaded wikidata resolution from cache.
185 / 490 resolved via Wikidata
Loaded SPARQL info from cache.
instance_of/field_of_work found for 185 / 185 QIDs


---

#### Classifying provider typologies based on Wikidata metadata

To assign each provider to a standardized typology, a cascading rule-based classification function is applied to the metadata previously retrieved.

* **Typology Categories**: Providers are mapped to one of 8 distinct categories (e.g., `library/archive`, `art/history museum`, `audiovisual/film archive`) or marked as `other`.
* **Regex Rule Matching**: A predefined set of regular expressions (`INSTITUTION_RULES`) scans the textual metadata for specific English and localized keywords (e.g., `museum`, `biblioteca`, `university`).
* **Classification Hierarchy**: The algorithm evaluates the metadata in a specific order of precedence to resolve conflicts:
  1. The raw provider name provided by Europeana.
  2. The direct entity description retrieved via the Wikidata REST API.
  3. The structural properties (`instance_of` and `field_of_work`) retrieved via the SPARQL endpoint.
* **Manual Overrides**: For ambiguous providers where automated entity resolution fails or returns conflicting data, a hardcoded dictionary (`CONFIRMED_FIXES`) forces the correct categorization.

In [ ]:
# Provider Typologies Rules and Classification
INSTITUTION_RULES = [
    (r"film archive|cin[ée]math[èe]que|kinemathek|audiovisual archive|\baudiovisual\b|\bcinema\b|cinecitt[aà]|"
     r"moving image|sound\s*(&|and)\s*vision|film museum|film institute", "audiovisual/film archive"),
    (r"\bmusic\b|musical|ethnomusicology|conservator(y|io|oire)|philharmonic|philharmonie|concert hall",
     "audiovisual/film archive"),

    (r"national library|public library|research library|\blibrar", "library/archive"),
    (r"\barchiv|\barquiv|national archives|repositor", "library/archive"),
    (r"bibliotec|bibliothek", "library/archive"),

    (r"natural history museum|science museum|herbarium|botanic|\bzoo\b|planetarium|aquarium|"
     r"geological survey|tropical.{0,15}research", "natural history/science institution"),

    (r"art museum|history museum|national museum|encyclopedic museum|archaeological museum|"
     r"specialized museum|military museum|open-air museum|\bmuseum\b|\bmuseo\b|mus[ée]e|\bmuseu\b|"
     r"museen|\bgallery\b|art collection|photograph|\bcastle\b|\bpalace\b|historical monument|"
     r"epigraphic|archaeolog|archeolog", "art/history museum"),

    (r"broadcaster|television|radio station|newspaper|press agency|publishing house",
     "media/broadcast organization"),

    (r"\buniversity\b|research institute|academy of sciences|research cent(er|re)|\bresearch\b|"
     r"\blaboratory\b|laboratoire", "academic/research institution"),

    (r"government agency|ministry|municipality|public administration|state institution|\bgovernment\b|"
     r"gobierno|ajuntament|gemeente|province of|heritage agency|superintendence|\bcourt\b",
     "government/administrative body"),
]
_compiled_institution_rules = [(re.compile(pat, re.IGNORECASE), name) for pat, name in INSTITUTION_RULES]


def classify_text(text):
    if not text:
        return None
    for pattern, category in _compiled_institution_rules:
        if pattern.search(text):
            return category
    return "other"


def classify_from_sparql(qid):
    info = sparql_info.get(qid, {"instance_of": [], "field_of_work": []})

    fow_category = classify_text(" | ".join(info["field_of_work"]))
    if fow_category not in (None, "other"):
        return fow_category

    io_category = classify_text(" | ".join(info["instance_of"]))
    if io_category not in (None, "other"):
        return io_category

    if not info["field_of_work"] and not info["instance_of"]:
        return None
    return "other"


def classify_provider(name):
    name_category = classify_text(name)
    wd_info = wikidata_resolution.get(name)
    desc_category = classify_text(wd_info.get("description")) if wd_info else None

    if name_category not in (None, "other") and desc_category not in (None, "other") and name_category != desc_category:
        # disagreement -- flag for review rather than silently picking one
        return "other", "name_description_conflict"
    if name_category not in (None, "other"):
        return name_category, "provider_name"
    if desc_category not in (None, "other"):
        return desc_category, "wikidata_description"

    if wd_info:
        sparql_category = classify_from_sparql(wd_info.get("qid"))
        if sparql_category not in (None, "other"):
            return sparql_category, "wikidata_sparql_fallback"
        if desc_category == "other" or sparql_category == "other" or name_category == "other":
            return "other", "unmatched"

    if name_category == "other":
        return "other", "unmatched"

    return None, None

classification_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider(name)
    classification_rows.append({"provider": name, "provider_category": category, "classification_source": source})

classification_df = pd.DataFrame(classification_rows)
print(classification_df["provider_category"].value_counts(dropna=False))
print()
print(classification_df["classification_source"].value_counts(dropna=False))


provider_category
library/archive                        201
art/history museum                     139
other                                   78
academic/research institution           29
audiovisual/film archive                17
government/administrative body          14
natural history/science institution      9
media/broadcast organization             3
Name: count, dtype: int64

classification_source
provider_name                373
unmatched                     71
wikidata_description          29
wikidata_sparql_fallback      10
name_description_conflict      7
Name: count, dtype: int64


In [ ]:
# Provider Classification and Manual Override
CONFIRMED_FIXES = {
    # Entity-resolution errors (Wikidata matched the wrong entity or a misleading description)
    "Brixiana": "library/archive",
    "Paul Van Riel": "other",

    # Government heritage-protection agencies mismatched by museum/archaeology keywords
    "Cultural Heritage Agency of the Netherlands": "government/administrative body",
    "Historical Monuments: Regional Conservation": "government/administrative body",
    "Ministry of Culture and Communication, Regional Archaeology Service": "government/administrative body",

    # Archaeology/epigraphic RESEARCH institutes mismatched as museums (the "archaeolog"/"epigraphic"
    "German Archaeological Institute": "academic/research institution",
    "Epigraphic Database Roma": "academic/research institution",
    "Epigraphic Dabatase Bari": "academic/research institution",
    "University Institute for Research in Iberian Archeology": "academic/research institution",
    "CISA -Interdipartimental Center for Archaeology": "academic/research institution",

    # "Conservatory" false-friend matches
    "National Conservatory of Arts and Crafts": "academic/research institution",
    "Conservatory of the Gironde Estuary": "government/administrative body",

    # Science/technology museums swept into the generic art/history museum bucket
    "Museon-Omniversum": "natural history/science institution",
    "Technoseum": "natural history/science institution",
    "Zoological Research Museum Koenig": "natural history/science institution",
}

review_rows = []
for name in distinct_providers["provider"]:
    if name in CONFIRMED_FIXES:
        continue

    category, source = classify_provider(name)
    if category in (None, "other"):
        wd = wikidata_resolution.get(name, {})
        item_count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
        review_rows.append({
            "provider": name,
            "description_en": wd.get("description_en"),
            "category": category,
            "source": source,
            "count": item_count,
        })

new_review_df = pd.DataFrame(review_rows).sort_values("count", ascending=False)

review_path = DATA_DIR / "csv" / "manual_review.csv"
if review_path.exists():
    existing_df = pd.read_csv(review_path)
    existing_categories = dict(zip(existing_df["provider"], existing_df["manual_category"]))
    new_review_df["manual_category"] = new_review_df["provider"].map(existing_categories).fillna("")
else:
    new_review_df["manual_category"] = ""

new_review_df.to_csv(review_path, index=False)
print(f"{len(new_review_df)} providers needed review, {new_review_df['count'].sum():,} total items")
filled = (new_review_df["manual_category"] != "").sum()

review_df = pd.read_csv(DATA_DIR / "csv" / "manual_review.csv")
review_df["manual_category"] = review_df["manual_category"].fillna("")
manual_overrides = {
    row["provider"]: row["manual_category"].strip()
    for _, row in review_df.iterrows()
    if row["manual_category"].strip()
}
print(f"{len(manual_overrides)} manual overrides loaded from CSV")
print(f"{len(CONFIRMED_FIXES)} confirmed fixes hard-coded")

def classify_provider_final(name):
    if name in CONFIRMED_FIXES:
        return CONFIRMED_FIXES[name], "confirmed_fix"
    if name in manual_overrides:
        return manual_overrides[name], "manual_override"
    return classify_provider(name)

final_categories = {name: classify_provider_final(name)[0] for name in distinct_providers["provider"]}

top_providers_df["provider_category"] = top_providers_df["provider"].map(final_categories)
top_providers_df["provider_category"] = top_providers_df["provider_category"].fillna("unresolved")

category_by_country = (
    top_providers_df.groupby(["country", "provider_category"])["count"]
    .sum()
    .unstack(fill_value=0)
)

category_by_country = category_by_country.merge(
    totals_df.set_index("country")["total_items"], left_index=True, right_index=True
)

sample_total = category_by_country.drop(columns="total_items").sum(axis=1)

category_share = category_by_country.drop(columns="total_items").div(
    category_by_country["total_items"], axis=0
)

category_share["coverage"] = sample_total / category_by_country["total_items"]

category_share = (category_share * 100).round(2)
category_share = category_share.reset_index()

category_share

audit_rows = []
for name in distinct_providers["provider"]:
    category, source = classify_provider_final(name)
    count = top_providers_df.loc[top_providers_df["provider"] == name, "count"].sum()
    country = distinct_providers.loc[distinct_providers["provider"] == name, "country"].iloc[0]
    audit_rows.append({
        "provider": name,
        "country": country,
        "count": count,
        "provider_category": category if category else "unresolved",
        "classification_source": source if source else "unresolved",
    })

cat_df = pd.DataFrame(audit_rows).sort_values(["country", "count"], ascending=[True, False])
cat_df.to_csv(DATA_DIR / "csv" / "providers_classified_complete.csv", index=False)

pd.set_option("display.max_rows", None)
cat_df.head(30)


77 providers needed review, 1,834,363 total items
77 manual overrides loaded from CSV
15 confirmed fixes hard-coded


,provider,country,count,provider_category,classification_source
100,National Library of France,france,2999310,library/archive,provider_name
101,Media Library of Architecture and Heritage,france,510877,library/archive,provider_name
102,Natural History Museum in Paris,france,502160,natural history/science institution,provider_name
103,Ministry of Culture,france,173279,government/administrative body,provider_name
104,Historical Monuments: Regional Conservation,france,128192,government/administrative body,confirmed_fix
105,"Ministry of Culture and Communication, Regiona...",france,110142,government/administrative body,confirmed_fix
106,National Audiovisual Institute France,france,54491,audiovisual/film archive,provider_name
107,Interuniversity Health Library,france,47992,library/archive,provider_name
108,Palais Galliera - Musée de la Mode de la Ville...,france,44495,art/history museum,provider_name
109,Mobilier National Collections,france,24284,art/history museum,manual_override


---

#### Visualizing RsQ04
To answer RsQ04, an interactive visualization is generated to explore the data at the national level.

* **Chart Type**: Horizontal Bar Chart.
* **Visual Encoding**: The top 20 providers for a selected country are displayed on the left. The x-axis represents the percentage of that country's total items contributed by each provider.
* **Interactivity**: A dropdown menu allows dynamically switching between the 6 target countries.

In [ ]:
# Visualization Top Providers typologies per country
TOP_N_DISPLAY = 10

provider_bar_state = {}
for country in totals_df["country"]:
    country_total = totals_df.loc[totals_df["country"] == country, "total_items"].iloc[0]
    top10 = (
        top_providers_df[top_providers_df["country"] == country]
        .sort_values("count", ascending=False)
        .head(TOP_N_DISPLAY)
        .copy()
    )
    top10["pct"] = top10["count"] / country_total * 100
    top10 = top10.sort_values("pct", ascending=True)

    provider_bar_state[country] = {
        "labels": list(top10["provider"]),
        "values": list(top10["pct"]),
        "hover": [
            f"<b>{p}</b><br>{c:,} items<br>{pct:.1f}% of {country.capitalize()}\'s total"
            for p, c, pct in zip(top10["provider"], top10["count"], top10["pct"])
        ],
    }

default_country = totals_df["country"].iloc[0]
default_state = provider_bar_state[default_country]

fig_bar = go.Figure(go.Bar(
    x=default_state["values"],
    y=default_state["labels"],
    orientation="h",
    marker=dict(color=COUNTRY_COLORS.get(default_country, "#CCCCCC")),
    customdata=default_state["hover"],
    hovertemplate="%{customdata}<extra></extra>",
    text=[f"{v:.1f}%" for v in default_state["values"]],
    textposition="outside",
    textfont=dict(color="black"),
    cliponaxis=False,
))

buttons = []
for country in totals_df["country"]:
    state = provider_bar_state[country]
    buttons.append(dict(
        label=country.capitalize(),
        method="update",
        args=[
            {
                "x": [state["values"]],
                "y": [state["labels"]],
                "customdata": [state["hover"]],
                "text": [[f"{v:.1f}%" for v in state["values"]]],
                "marker.color": [COUNTRY_COLORS.get(country, "#CCCCCC")],
            },
            {"title.text": f"Top {TOP_N_DISPLAY} providers"},
        ],
    ))

fig_bar.update_layout(
    title=dict(text=f"Top {TOP_N_DISPLAY} providers", x=0.5, xanchor="center"),
    xaxis_title="% of country's total items",
    height=700,
    margin=dict(l=250, r=60, t=120, b=40),  # generous left margin for long provider names
    font=dict(family="Roboto", size=14, color="#444"),
    updatemenus=[dict(
        type="dropdown", direction="down", x=1.0, y=1.15, xanchor="right",
        buttons=buttons, showactive=True,
    )],
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#EEEEEE", zeroline=False),
    yaxis=dict(showgrid=False),
)

fig_bar.show()


#### Visualizing RsQ05
To answer RsQ05,  an interactive small-multiples visualization is generated at the national level.

* **Provider Concentration by Country (Treemap Grid)**
  * **Chart Type**: Interactive treemap grid (`plotly.graph_objects.Treemap`), arranged in a 2×3 subplot layout, one treemap per country.
  * **Visual Encoding**: Each country's treemap is split into three segments — the single largest provider ("Top 1"), the next four largest combined ("Rank 2–5"), and all remaining providers ("Other") — sized by their share of that country's total item volume. Segment color intensity (dark to light) reinforces the concentration gradient, from most to least dominant.
* **Ordering**: Countries are arranged from most to least concentrated (by `pct_top1_provider`), so the grid itself reads as a ranking, left to right,  top to bottom.
* **Interactivity**: Hovering over any segment reveals its exact percentage share.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plot_df = concentration_df.sort_values("pct_top1_provider", ascending=False).reset_index(drop=True)
totals_lookup = totals_df.set_index("country")["total_items"].to_dict()

segment_colors = ["#203464", "#4b74a0", "#679e91"]
top_n = 5  

def get_rank_names(country, start, end):
    df = providers_by_country[country].sort_values("count", ascending=False).reset_index(drop=True)
    names = df.iloc[start-1:end]["provider"].tolist()
    return "<br>".join(f"• {n}" for n in names)

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "treemap"}]*3, [{"type": "treemap"}]*3],
    subplot_titles=[
        f"{c}<br><span style='font-size:11px;color:#888'>{totals_lookup.get(c, 0):,} items</span>"
        for c in plot_df["country"]
    ],
    horizontal_spacing=0.03,
    vertical_spacing=0.16,
)

for idx, row in plot_df.iterrows():
    r = idx // 3 + 1
    c = idx % 3 + 1

    top1 = row["pct_top1_provider"]
    top_n_minus_1 = row[f"pct_top{top_n}_providers"] - top1
    others = 100 - row[f"pct_top{top_n}_providers"]

    rank2_5_names = get_rank_names(row["country"], 2, top_n)

    labels = [row["top_provider"], f"Rank 2–{top_n}", "Other providers"]
    # customdata aligned with labels: extra text shown only for the "Rank 2-5" segment
    customdata = [[row["top_provider"]], [rank2_5_names], [""]]

    fig.add_trace(
        go.Treemap(
            labels=labels,
            parents=["", "", ""],
            values=[top1, top_n_minus_1, others],
            marker=dict(colors=segment_colors, line=dict(width=1, color="white")),
            texttemplate="%{label}<br>%{value:.0f}%",
            textfont=dict(size=12, color="white"),
            customdata=customdata,
            hovertemplate="<b>%{label}</b>: %{value:.1f}%<br>%{customdata[0]}<extra></extra>",
            textposition="middle center",
        ),
        row=r, col=c,
    )

fig.update_layout(
    title=dict(text="Provider concentration by country", x=0.5, xanchor="center", font=dict(size=20)),
    height=650, width=1050,  # can shrink height slightly since no legend row
    margin=dict(t=90, b=30, l=20, r=20),  # less bottom margin needed
    paper_bgcolor="white",
    font=dict(family="Roboto, sans-serif"),
)

for annotation in fig["layout"]["annotations"]:
    annotation["font"] = dict(size=14, color="#333333")

fig.show()

#### Visualizing RsQ06

To answer RsQ06, an interactive geographic visualization is generated at the provider level.

* **Where Digitization Happens: Top Providers by Location (Interactive Map)**
  * **Chart Type**: Interactive scatter map (`plotly.express.scatter_map`), rendered over a light basemap (`carto-positron`), centered and zoomed on Europe.
  * **Visual Encoding**: Each point represents one of the top 20 providers per country. Marker color encodes country while marker size is mapped from item count via a square-root scale (`size_mapped`), compressing the heavy-tailed volume distribution so that both very large and very small providers remain visible on the same map.
* **Two Display Modes**: The visualization toggles between two trace sets via button controls — "Show all providers" displays every resolved provider as an unlabeled point, while "Top 5 per country" isolates each country's five largest providers, adding text labels with provider names directly on the map.

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go  

provider_geo_final = provider_geo_final.sort_values("count", ascending=False)

provider_geo_final["rank_in_country"] = (
    provider_geo_final.groupby("country")["count"]
    .rank(method="first", ascending=False)
)
provider_geo_final["is_top5"] = provider_geo_final["rank_in_country"] <= 5

min_size, max_size = 8, 40
c = provider_geo_final["count"]
provider_geo_final["size_mapped"] = min_size + (np.sqrt(c) - np.sqrt(c.min())) / (np.sqrt(c.max()) - np.sqrt(c.min())) * (max_size - min_size)

fig = px.scatter_map(
    provider_geo_final,
    lat="latitude",
    lon="longitude",
    color="country",
    size="size_mapped",
    size_max=max_size,     # <-- prevents Plotly re-scaling size_mapped again
    hover_name="provider",
    hover_data={"city": True, "count": ":,", "size_mapped": False, "source": False,
                "latitude": False, "longitude": False},
    zoom=3.3,
    center=dict(lat=44, lon=-3),
    map_style="carto-positron",
    title="Where Digitization Happens: Top Providers by Location",
    color_discrete_map=COUNTRY_COLORS,
)

n_all_traces = len(fig.data)  # one trace per country from the "all" view

# --- Style + hover for the BASE traces only (use selector, not a blanket update_traces) ---
fig.update_traces(
    selector=lambda t: t.name in COUNTRY_COLORS,   # only the px-built base traces
    marker=dict(opacity=0.75),
    hovertemplate=(
        "<b>%{hovertext}</b><br>" +
        "City: %{customdata[0]}<br>" +
        "Items in Europeana: %{customdata[1]:,}<br>" +
        "<extra></extra>"
    ),
)

# --- Top-5-per-country traces (separate, added after) ---
top5 = provider_geo_final[provider_geo_final["is_top5"]]

for country in top5["country"].unique():
    sub = top5[top5["country"] == country]
    fig.add_trace(go.Scattermap(
        lat=sub["latitude"], lon=sub["longitude"],
        mode="markers+text",
        marker=dict(size=sub["size_mapped"], color=COUNTRY_COLORS[country], opacity=0.9),
        text=sub["provider"],
        textposition="top right",
        textfont=dict(size=9, color="#222"),
        name=country,
        hovertext=sub["provider"],
        hovertemplate="<b>%{hovertext}</b><br>Items: %{customdata:,}<extra></extra>",
        customdata=sub["count"],
        visible=False,
        showlegend=False,
    ))

n_total = len(fig.data)
top5_indices = list(range(n_all_traces, n_total))

# --- Hover label colors, applied per trace by name (safe now, runs after all traces exist) ---
def hex_to_rgba(hex_color, alpha=0.85):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"

def readable_font_color(hex_color):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
    return "#222222" if luminance > 0.6 else "#FBF9F5"

for trace in fig.data:
    country = trace.name
    base_color = COUNTRY_COLORS.get(country, "#FBF9F5")
    trace.hoverlabel = dict(
        bgcolor=hex_to_rgba(base_color, 0.75),
        bordercolor=base_color,
        font=dict(color=readable_font_color(base_color)),
    )

fig.update_layout(
    height=650, width=950,
    legend_title="Country",
    margin=dict(l=10, r=10, t=60, b=10),
    title=dict(x=0.5, xanchor="center", font=dict(size=20, family="Roboto", color="#333")),
    font=dict(family="Roboto", size=14, color="#444"),
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.5, y=1.08, xanchor="center",
            showactive=True,
            buttons=[
                dict(
                    label="Show all providers",
                    method="update",
                    args=[{"visible": [True] * n_all_traces + [False] * len(top5_indices)}],
                ),
                dict(
                    label="Top 5 per country",
                    method="update",
                    args=[{"visible": [False] * n_all_traces + [True] * len(top5_indices)}],
                ),
            ],
        )
    ],
)

fig.show()

#### Visualizing RsQ07
To explore the providers' categories, two visualizations are generated at both the aggregate and national levels.

* **Overall Typology Distribution (Pie Chart)**
  * **Chart Type**: Pie Chart (`plotly.graph_objects.Pie`).
  * **Visual Encoding**: Aggregates the item volumes across all 6 countries to show the macro-level distribution of provider typologies.

* **Typology Contribution by Country (Sunburst Chart)**
  * **Chart Type**: Multi-level Sunburst Chart (`plotly.graph_objects.Sunburst`).
  * **Visual Encoding**: The inner ring represents the proportional contribution of each country to the total item volume of the 6-country dataset. The outer ring breaks down each country's slice into its specific provider typologies.

In [ ]:
# Visualization Category share within and across countries
# (a) category share WITHIN each country -- against that country's true total item count
category_by_country = cat_df.groupby(["country", "provider_category"])["count"].sum().unstack(fill_value=0)
category_by_country = category_by_country.merge(
    totals_df.set_index("country")["total_items"], left_index=True, right_index=True
)

sample_total = category_by_country.drop(columns="total_items").sum(axis=1)

category_share_by_country = category_by_country.drop(columns="total_items").div(
    category_by_country["total_items"], axis=0
)
category_share_by_country["coverage"] = sample_total / category_by_country["total_items"]
category_share_by_country = (category_share_by_country * 100).round(1).reset_index()

category_share_by_country.to_csv(DATA_DIR / "csv" / "category_share_by_country.csv", index=False)
print("Category share within each country (%):")
category_share_by_country

# (b) each category's share of the OVERALL six-country total -- "contribution to Europeana" in aggregate
overall_total = cat_df["count"].sum()

overall_category_share = (
    cat_df.groupby("provider_category")["count"]
    .sum()
    .reset_index()
    .rename(columns={"count": "items"})
)
overall_category_share["pct_of_europeana_total"] = (overall_category_share["items"] / overall_total * 100).round(1)
overall_category_share = overall_category_share.sort_values("pct_of_europeana_total", ascending=False)

overall_category_share.to_csv(DATA_DIR / "csv" / "category_share_overall.csv", index=False)
print(f"\nCategory share of the full six-country total ({overall_total:,} items):")
overall_category_share

plot_df = overall_category_share.sort_values("pct_of_europeana_total", ascending=False)

fig_overall = go.Figure(go.Pie(
    labels=plot_df["provider_category"],
    values=plot_df["items"],
    marker=dict(colors=[CATEGORY_COLORS.get(c, "#D1D5DB") for c in plot_df["provider_category"]]),
    textinfo="none",
    textfont=dict(color="black"),
    insidetextorientation="horizontal",
    hovertemplate="<b>%{label}</b><br>%{value:,} items<br>%{percent} of the six-country total<extra></extra>",
))

fig_overall.update_layout(
    title=dict(text="Category share of Europeana's total items (six countries combined)", x=0.5, xanchor="center"),
    height=600,
    margin=dict(l=40, r=40, t=90, b=40),
    font=dict(family="Roboto", size=14, color="#444"),
    showlegend=True,
    legend=dict(
        title=dict(text="Category", font=dict(size=18)),
        font=dict(size=16),
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    uniformtext=dict(minsize=10, mode="hide"),
)

fig_overall.show()


Category share within each country (%):

Category share of the full six-country total (30,879,627 items):


In [70]:
# Visualization Country contribution to Europeana, by provider category
if 'cat_df' not in globals():
    import pandas as pd
    from pathlib import Path
    cat_df = pd.read_csv(Path('data/csv/providers_classified_complete.csv'))

COUNTRY_COLORS = {
    "netherlands": "#a180ad",
    "france": "#1f7f95",
    "portugal": "#f4a64e",
    "italy": "#90BE6D",
    "germany": "#feda15",
    "spain": "#bb521f",
}

CATEGORY_COLORS = {
    "audiovisual/film archive": "#80CBC4",
    "art/history museum": "#9FA8DA",
    "natural history/science institution": "#CE93D8",
    "library/archive": "#90CAF9",
    "academic/research institution": "#FFCC80",
    "media/broadcast organization": "#EF9A9A",
    "government/administrative body": "#E8A0BE",
    "other": "#CFCFCF",
    "unresolved": "#9E9E9E",
}

COUNTRY_ISO = {
    "netherlands": "NL",
    "france": "FR",
    "portugal": "PT",
    "italy": "IT",
    "germany": "DE",
    "spain": "ES",
}

FONT_FAMILY = "Roboto, sans-serif"

provider_counts_by_country = cat_df.groupby("country").size()
total_providers = len(cat_df)

sunburst_country = cat_df.groupby("country")["count"].sum().reset_index()
sunburst_cat = cat_df.groupby(["country", "provider_category"])["count"].sum().reset_index()

ids, labels, parents, values, colors, hover_text, hoverinfo_list = [], [], [], [], [], [], []

def add_node(node_id, label, parent, value, color, hover, hoverinfo="all"):
    ids.append(node_id)
    labels.append(label)
    parents.append(parent)
    values.append(value)
    colors.append(color)
    hover_text.append(hover)
    hoverinfo_list.append(hoverinfo)

add_node("europeana", "Europeana", "", sunburst_country["count"].sum(), "#FFFFFF", "", hoverinfo="skip")

for _, row in sunburst_country.iterrows():
    country = row["country"]
    item_count = row["count"]
    n_providers = provider_counts_by_country.get(country, 0)
    provider_pct = n_providers / total_providers * 100
    add_node(
        country, COUNTRY_ISO.get(country, ""), "europeana", item_count,
        COUNTRY_COLORS.get(country, "#CCCCCC"),
        f"<b>{country.capitalize()}</b><br>{item_count:,} items<br>{provider_pct:.1f}% of all providers",
    )

for _, row in sunburst_cat.iterrows():
    country, cat, count = row["country"], row["provider_category"], row["count"]
    country_total = sunburst_country.loc[sunburst_country["country"] == country, "count"].iloc[0]
    cat_pct_of_country = count / country_total * 100
    add_node(
        f"{country}-{cat}", "", country, count,
        CATEGORY_COLORS.get(cat, "#D1D5DB"),
        f"<b>{cat}</b><br>{count:,} items<br>{cat_pct_of_country:.1f}% of {country.capitalize()}",
    )

FIG_WIDTH = 1250
FIG_HEIGHT = 800

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(colors=colors),
    customdata=hover_text,
    hovertemplate="%{customdata}<extra></extra>",
    hoverinfo=hoverinfo_list,
    insidetextorientation="horizontal",
    textinfo="label",
    textfont=dict(family="Roboto", color="black", size=22),
    insidetextfont=dict(family="Roboto", color="black", size=22),
    domain=dict(x=[0.02, 0.73], y=[0.02, 0.98]),
))

categories_present = sunburst_cat["provider_category"].unique()
for cat in categories_present:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=10, color=CATEGORY_COLORS.get(cat, "#D1D5DB")),
        name=cat,
        showlegend=True,
        hoverinfo="skip",
    ))

fig.update_layout(
    title=dict(text="Country contribution to Europeana, by provider category", x=0.5, xanchor="center"),
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(t=80, l=20, r=20, b=20),
    font=dict(family="Roboto", color="#444"),
    showlegend=True,
    legend=dict(
        title=dict(text="Category", font=dict(family="Roboto", color="#444", size=22)),
        font=dict(family="Roboto", color="#444", size=16),
        yanchor="middle", y=0.5, xanchor="left", x=0.76,
    ),
    hoverlabel=dict(font=dict(family="Roboto", size=16)),
    uniformtext=dict(minsize=7, mode="hide"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)

fig.show()

## RQ03 How do countries compare in terms of Europeana quality standards and dataset openness?

### RsQ08 To what extent do countries meet Europeana's quality standards?

The final part of our analysis asks how well the digitization efforts of the countries under study meet Europeana's own publishing standards. To assess the quality of contributed records, Europeana classifies each item according to two complementary frameworks: Content Tier and Metadata Tier. The Content Tier measures content quality and reusability, taking into account not only the quality of the digital resource itself but also the rights statements and licenses applied to it. The scale ranges from 0 to 4, with Europeana considering CT2–CT4 as meeting its minimum publishing quality requirements. The Metadata Tier measures the completeness and richness of a record's descriptive metadata, scored across three criteria (language tagging, enabling elements, and contextual class links). Europeana's documentation formally defines three levels, A through C, each with increasing thresholds; a "0" value also appears in the live API data and in Europeana's own example datasets, functioning as the lowest tier for records that don't meet the minimum bar. We collected this data from Europeana API.

### RsQ09 To what extent are countries' cultural datasets open and reusable?

This subquestion examines how freely each country's digitized cultural heritage can be reused. Within Europeana's quality framework, reusability is directly linked to content quality: the higher Content Tiers (CT3 and CT4) require rights statements that explicitly permit reuse. Because licensing is the result of institutional and national policy choices rather than technical constraints, it provides an informative perspective on the final question of our study: how countries position themselves with respect to their own cultural heritage and to the idea of a shared European cultural commons that Europeana was created to promote.


#### Querying the Europeana API for quality score and reusability parameter

Two sets of country-level counts are collected from Europeana's Search API:

* **Quality Score Counts**: `fetch_country_facets()` queries each country retrieving distribution counts for `contentTier` and `metadataTier`.
* **Reusability Counts**: `fetch_reusability()` issues one request per country per `reusability` value, since reusability is implemented as a request parameter rather than a facet value, and cannot be retrieved alongside the other facets in a single call. Europeana groups the 60+ raw rights-statement values found in the `RIGHTS` facet into three categories: `open` (Public Domain Mark, CC0, CC BY, CC BY-SA), `restricted` (CC BY-NC and similar licenses with conditions), and `permission` (rights reserved, reuse requires explicit authorization).

In [ ]:
FACETS = "contentTier,metadataTier"
REUSABILITY_VALUES = ["open", "restricted", "permission"]

# Europeana's own dividing line: tier 1 = does not meet the quality criteria, tiers 2/3/4 = meets it. 
QUALITY_CONTENT_TIERS = {"2", "3", "4"}
QUALITY_METADATA_TIERS = {"A", "B", "C"}

def fetch_country_facets(country):
    return europeana_search(
        query="*",
        qf=[f'COUNTRY:"{country}"'],
        facet=FACETS,
        rows=0,
        **{"f.DATA_PROVIDER.facet.limit": 500},
    )

def fetch_reusability(country):
    counts = {}
    for value in REUSABILITY_VALUES:
        result = europeana_search(
            query="*",
            qf=[f'COUNTRY:"{country}"'],
            rows=0,
            reusability=value,
        )
        counts[value] = result.get("totalResults", 0)
    return counts

json_dir = DATA_DIR / "json"
json_dir.mkdir(parents=True, exist_ok=True)

facets_path = json_dir / "country_quality_score.json"
reusability_path = json_dir / "country_reusability.json"

if facets_path.exists() and reusability_path.exists():
    with open(facets_path) as f:
        country_facets = json.load(f)
    with open(reusability_path) as f:
        country_reusability = json.load(f)
    print("Loaded cached quality and reusability data")
else:
    country_facets = {country: fetch_country_facets(country) for country in TARGET_COUNTRIES}
    country_reusability = {country: fetch_reusability(country) for country in TARGET_COUNTRIES}

    with open(facets_path, "w") as f:
        json.dump(country_facets, f, indent=2)
    with open(reusability_path, "w") as f:
        json.dump(country_reusability, f, indent=2)
    print(f"Fetched and saved quality data to {facets_path}")
    print(f"Fetched and saved reusability data to {reusability_path}")


Loaded cached quality and reusability data


#### Computing quality and openness metrics

From the raw quality and reusability data collected above, this section derives the per-country metrics.

* **`facet_dict()`**: extracts a single named facet's label→count mapping from a country's raw API response, used as a helper by all functions below.
* **`quality_summary()`**: computes aggregate quality metrics per country — the share of items meeting Europeana's content-tier threshold (`pct_contentTier_2plus`, tiers 2–4) and metadata-tier threshold (`pct_metadataTier_ABC`, tiers A–C) — alongside the share at the lowest tier for each (`pct_contentTier_0`, `pct_metadataTier_0`).
* **`contenttier_breakdown()`**: disaggregates content tier into its individual levels (0–4) per country, since the aggregate.
* **`metadatatier_breakdown()`**: disaggregates metadata tier into its individual levels (0, A, B, C) per country.
* **`openness_summary()`**: converts the raw reusability counts (`open`, `restricted`, `permission`) into per-country percentages of each category.

In [ ]:
def facet_dict(country_data, facet_name):
    for facet in country_data["facets"]:
        if facet["name"] == facet_name:
            return {f["label"]: f["count"] for f in facet["fields"]}
    return {}


def quality_summary(facets_path=facets_path):
    with open(facets_path) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        ct = facet_dict(data, "contentTier")
        mt = facet_dict(data, "metadataTier")

        content_ok = sum(v for k, v in ct.items() if k in QUALITY_CONTENT_TIERS)
        metadata_ok = sum(v for k, v in mt.items() if k in QUALITY_METADATA_TIERS)

        rows.append({
            "country": country,
            "total_items": total,
            "pct_contentTier_2plus": round(100 * content_ok / total, 1),
            "pct_contentTier_0": round(100 * ct.get("0", 0) / total, 1),
            "pct_metadataTier_ABC": round(100 * metadata_ok / total, 1),
            "pct_metadataTier_0": round(100 * mt.get("0", 0) / total, 1),
        })

    return pd.DataFrame(rows).sort_values("pct_contentTier_2plus", ascending=False)


def contenttier_breakdown(facets_path=facets_path):
    with open(facets_path) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        ct = facet_dict(data, "contentTier")

        row = {"country": country, "total_items": total}
        for tier in ["0", "1", "2", "3", "4"]:
            row[f"pct_tier_{tier}"] = round(100 * ct.get(tier, 0) / total, 1)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("pct_tier_4", ascending=False)


tier_breakdown_df = contenttier_breakdown()
tier_breakdown_df


def metadatatier_breakdown(facets_path=facets_path):
    with open(facets_path) as f:
        raw = json.load(f)

    rows = []
    for country, data in raw.items():
        total = data["totalResults"]
        mt = facet_dict(data, "metadataTier")

        row = {"country": country, "total_items": total}
        for tier in ["0", "A", "B", "C"]:
            row[f"pct_metadataTier_{tier}"] = round(100 * mt.get(tier, 0) / total, 1)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("pct_metadataTier_C", ascending=False)


metadata_breakdown_df = metadatatier_breakdown()
metadata_breakdown_df


def openness_summary(reusability_results):
    rows = []
    for country, counts in reusability_results.items():
        total = sum(counts.values())
        rows.append({
            "country": country,
            "pct_open": round(100 * counts["open"] / total, 1) if total else None,
            "pct_restricted": round(100 * counts["restricted"] / total, 1) if total else None,
            "pct_permission": round(100 * counts["permission"] / total, 1) if total else None,
        })
    return pd.DataFrame(rows)

#### Visualizing RsQ08

To answer RsQ08 about quality metrics, a paired horizontal stacked-bar visualization is generated to show the internal composition behind each country's quality scores.

* **How Countries Meet Europeana's Quality Thresholds (Paired Stacked Bars)**
  * **Chart Type**: Two horizontal stacked bar charts (`plotly.graph_objects.Bar`), placed side by side as subplots sharing a common y-axis.
  * **Visual Encoding**: Each bar represents one country's full tier distribution — content tier (0–4) on the left, metadata tier (0, A, B, C) on the right — with segment length proportional to the share of items at each tier. Each dimension uses its own single-hue sequential color ramp (blue for content, orange for metadata), with color intensity increasing from lowest to highest tier, so darker segments always indicate higher quality within their respective scale.
* **Ordering**: Countries share the same order across both subplots (by `pct_tier_4`, content tier's top level), so a single horizontal row shows one country's full content and metadata profile side by side.
* **Interactivity**: Hovering over any segment reveals its exact percentage share; a shared, grouped legend distinguishes all content and metadata tier levels.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

content_colors = {
    "pct_tier_0": "#E8E8E8",
    "pct_tier_1": "#C6DBEF",
    "pct_tier_2": "#6BAED6",
    "pct_tier_3": "#2171B5",
    "pct_tier_4": "#08306B",
}
metadata_colors = {
    "pct_metadataTier_0": "#E8E8E8",
    "pct_metadataTier_A": "#FDD0A2",
    "pct_metadataTier_B": "#FD8D3C",
    "pct_metadataTier_C": "#A63603",
}

# Same country order on both subplots -- sort by a meaningful reference,
# e.g. content tier 4 share, so the row order tells its own story
order = tier_breakdown_df.sort_values("pct_tier_4", ascending=True)["country"]  # ascending so highest ends up at top in a horizontal bar
ct = tier_breakdown_df.set_index("country").loc[order]
mt = metadata_breakdown_df.set_index("country").loc[order]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Content tier composition", "Metadata tier composition"),
    shared_yaxes=True,
    horizontal_spacing=0.08,
)

for col, color in content_colors.items():
    tier_label = col.replace("pct_tier_", "Content – Tier ")
    fig.add_trace(
        go.Bar(y=ct.index, x=ct[col], orientation="h",
               name=tier_label, marker_color=color, legendgroup="content"),
        row=1, col=1
    )

for col, color in metadata_colors.items():
    tier_label = col.replace("pct_metadataTier_", "Metadata – Tier ")
    fig.add_trace(
        go.Bar(y=mt.index, x=mt[col], orientation="h",
               name=tier_label, marker_color=color, legendgroup="metadata"),
        row=1, col=2
    )

fig.update_xaxes(title_text="% of items", row=1, col=1)
fig.update_xaxes(title_text="% of items", row=1, col=2)

fig.update_layout(
    barmode="stack",
    height=500, width=1150,
    title_text="How Countries Meet Europeana's Quality Thresholds",
    legend=dict(title="Tier", tracegroupgap=10),
)

fig.show()

#### Visualizing RsQ09

To show how each country's reusability breakdown compares, a single stacked bar chart is generated summarizing the `open`/`restricted`/`permission` split at a glance.

* **How Open Is Each Country's Digitized Heritage? (Stacked Bar Chart)**
  * **Chart Type**: Vertical stacked bar chart (`plotly.graph_objects.Bar`).
  * **Visual Encoding**: Each bar represents one country's full reusability distribution, with segment height proportional to the share of items in each of the three Europeana reusability categories. A green-to-red color progression (`open` → `restricted` → `permission`) signals the reuse-friendliness of each segment, from least to most restrictive.
* **Ordering**: Countries are sorted by `pct_open`, from most to least openly licensed, so the chart itself reads as a ranking.
* **Interactivity**: Hovering over any segment reveals its exact percentage share.

In [ ]:
import plotly.graph_objects as go

openness_df = openness_summary(country_reusability)
order = openness_df.sort_values("pct_open", ascending=False)["country"]
plot_df = openness_df.set_index("country").loc[order]


fig_openness = go.Figure()
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_open"],
                               name="Open", marker_color="#31A354"))
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_restricted"],
                               name="Restricted", marker_color="#FDAE6B"))
fig_openness.add_trace(go.Bar(x=plot_df.index, y=plot_df["pct_permission"],
                               name="Permission", marker_color="#DE2D26"))
fig_openness.update_layout(
    barmode="stack",
    title="How Open Is Each Country's Digitized Heritage?",
    yaxis_title="% of items",
    height=500, width=700,
)
fig_openness.show()